In [1]:
from pathlib import Path

import duckdb
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """
    Walk upward from the current working directory until we find
    the project folders that identify this repository.
    """
    for candidate in (start_path, *start_path.parents):
        has_data_folder = (candidate / "data").exists()
        has_notebooks_folder = (candidate / "notebooks").exists()

        if has_data_folder and has_notebooks_folder:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the notebook is opened inside market-intelligence-pipeline."
    )


project_root = find_project_root(Path.cwd().resolve())

raw_olist_dir = project_root / "data" / "raw" / "olist"

print(f"Project root: {project_root}")
print(f"Raw Olist data: {raw_olist_dir}")

def get_raw_file(file_name: str) -> Path:
    """
    Find a file anywhere inside data/raw/olist/.
    """
    matches = list(raw_olist_dir.rglob(file_name))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {file_name} inside {raw_olist_dir}"
        )

    return matches[0]

orders = pd.read_csv(
    get_raw_file("olist_orders_dataset.csv"),
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    get_raw_file("olist_order_items_dataset.csv"),
    parse_dates=[
        "shipping_limit_date",
    ],
)


print("orders shape:", orders.shape)
print("order_items shape:", order_items.shape)

orders.head()

Project root: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline
Raw Olist data: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\raw\olist
orders shape: (99441, 8)
order_items shape: (112650, 7)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [ ]:

order_items['gross_item_value'] = order_items['price'] + order_items['freight_value'] # create a column for total value of an item
# aggregate to get order level total value, item count per order, avg item value
order_value = (
    order_items
    .groupby('order_id',as_index=False)
    .agg(
        gross_order_value = ('gross_item_value','sum'),
        item_count = ('order_item_id','count'),
        avg_item_value = ('gross_item_value','mean'),
    )
)
print("order value shape:", order_value.shape)
order_value.head()
# we also could have had .assign() here, which returns a new dataframe with created or replaced columns
# order_value = (
#     order_items
#     .assign(
#         total_item_value=lambda df: ( # here assign defines a new column, but it doesn't modify order_items, it creates a temporary dataset
#             df["price"]
#             + df["freight_value"]
#         )
#     )
#     .groupby("order_id", as_index=False)
#     .agg(
#         gross_order_value=("total_item_value", "sum"),
#         item_count=("order_item_id", "count"),
#     )
# )

order value shape: (98666, 4)


,order_id,gross_order_value,item_count,avg_item_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,218.04


In [6]:
print("Rows in order_value:", len(order_value))
print("Unique order IDs:", order_value["order_id"].nunique())
print("Is order_id unique?", order_value["order_id"].is_unique)

Rows in order_value: 98666
Unique order IDs: 98666
Is order_id unique? True


In [ ]:
# DuckDB can automatically detect pandas dataframes and query them by their variable names
# pandas df --> DuckDB reads it as SQL table --> write SQL --> DuckDB returns the result as pandas df
# duckdb.sql("""SQL QUERY HERE""").df() --> this is a pandas dataframe

duckdb.sql("""
    SELECT
        42 AS number,
        'Hello' AS text
""").df()

,number,text
0,42,Hello


In [8]:
# a helper function that takes in sql query as a string and returns the query result as a pandas df
def run_sql(sql_query: str) -> pd.DataFrame:
    return duckdb.sql(sql_query).df()

In [9]:
query = """
SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(oi.order_item_id) AS matched_item_rows
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
"""
run_sql(query)

,joined_rows,distinct_orders,matched_item_rows
0,112650,98666,112650


In [10]:
query = """
SELECT
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp
FROM orders o
LEFT JOIN order_items oi
ON o.order_id = oi.order_id
WHERE oi.order_item_id IS NULL;
"""
run_sql(query)

,order_id,order_status,order_purchase_timestamp
0,8e24261a7e58791d10cb1bf9da94df5c,unavailable,2017-11-16 15:09:28
1,c272bcd21c287498b4883c7512019702,unavailable,2018-01-31 11:31:37
2,37553832a3a89c9b2db59701c357ca67,unavailable,2017-08-14 17:38:02
3,d57e15fb07fd180f06ab3926b39edcd2,unavailable,2018-01-08 19:39:03
4,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39
...,...,...,...
770,aaab15da689073f8f9aa978a390a69d1,unavailable,2018-01-16 14:27:59
771,3a3cddda5a7c27851bd96c3313412840,canceled,2018-08-31 16:13:44
772,a89abace0dcc01eeb267a9660b5ac126,canceled,2018-09-06 18:45:47
773,a69ba794cc7deb415c3e15a0a3877e69,unavailable,2017-08-23 16:28:04


In [11]:
query = """
WITH order_value AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS gross_order_value,
        COUNT(order_item_id) AS item_count
    FROM order_items
    GROUP BY order_id
)
SELECT * FROM order_value;
"""
run_sql(query)

,order_id,gross_order_value,item_count
0,e880a2a0f19dcb0f2ef51fed71ef34c9,74.62,2
1,e885304ec221dd09a24101f0dd920504,169.87,1
2,e88b6796c87cd2719ff0c038a8997d54,53.99,1
3,e88bd747b9aaa829eac8fcf95461862e,58.76,2
4,e88e5f8bf5cb7adfd9f6435dee68112f,97.83,1
...,...,...,...
98661,e85a5feeff7fbda800ceecfd2fb221e1,144.28,2
98662,e85cc7bf67b73e1700ee99b800496267,99.43,1
98663,e8606bd877aee06d98501a6436ee8576,47.35,1
98664,e865720063d35fe789029a6021f2df31,35.00,1


In [13]:
query = """
WITH order_value AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS gross_order_value,
        COUNT(order_item_id) AS item_count
    FROM order_items
    GROUP BY order_id
)
SELECT 
    o.order_id,
    o.order_status,
    ov.gross_order_value,
    ov.item_count
FROM orders o
LEFT JOIN order_value ov
ON o.order_id = ov.order_id
LIMIT 10;
"""
run_sql(query)

,order_id,order_status,gross_order_value,item_count
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,38.71,1
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,141.46,1
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,179.12,1
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,72.20,1
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,28.62,1
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,175.26,1
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,65.95,1
7,6514b8ad8028c9f2cc2374ded245783f,delivered,75.16,1
8,76c6e866289321a7c93b82b54852dc33,delivered,35.95,1
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,169.76,1


In [14]:
query = """
WITH order_value AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS gross_order_value,
        COUNT(order_item_id) AS item_count
    FROM order_items
    GROUP BY order_id
)
SELECT
    o.order_status,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    SUM(COALESCE(ov.gross_order_value, 0)) AS total_gross_order_value,
    AVG(COALESCE(ov.gross_order_value, 0)) AS average_gross_order_value
FROM orders o
LEFT JOIN order_value ov
ON o.order_id = ov.order_id
GROUP BY o.order_status
ORDER BY total_gross_order_value DESC;
"""
run_sql(query)

,order_status,distinct_orders,total_gross_order_value,average_gross_order_value
0,delivered,96478,1.541977e+07,159.826839
1,shipped,1107,1.771293e+05,160.008437
2,canceled,625,1.058857e+05,169.417152
3,processing,301,6.939411e+04,230.545216
4,invoiced,314,6.898875e+04,219.709395
5,unavailable,609,2.140490e+03,3.514762
6,approved,2,2.410800e+02,120.540000
7,created,5,0.000000e+00,0.000000


In [15]:
query = """
WITH order_value AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS gross_order_value,
        COUNT(order_item_id) AS item_count
    FROM order_items
    GROUP BY order_id
)
SELECT
    o.order_status,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    SUM(ov.gross_order_value) AS total_gross_order_value,
    AVG(ov.gross_order_value) AS average_gross_order_value
FROM orders o
LEFT JOIN order_value ov
ON o.order_id = ov.order_id
GROUP BY o.order_status
ORDER BY total_gross_order_value DESC;
"""
run_sql(query)

,order_status,distinct_orders,total_gross_order_value,average_gross_order_value
0,delivered,96478,1.541977e+07,159.826839
1,shipped,1107,1.771293e+05,160.153110
2,canceled,625,1.058857e+05,229.687028
3,processing,301,6.939411e+04,230.545216
4,invoiced,314,6.898875e+04,221.117788
5,unavailable,609,2.140490e+03,356.748333
6,approved,2,2.410800e+02,120.540000
7,created,5,NaN,NaN


In [16]:
query = """
WITH order_value AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS gross_order_value,
        COUNT(order_item_id) AS item_count
    FROM order_items
    GROUP BY order_id
)
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp) AS purchase_month,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    SUM(ov.gross_order_value) AS total_gross_order_value,
    AVG(ov.gross_order_value) AS average_gross_order_value,
    AVG(ov.item_count) AS average_item_count
FROM orders o
INNER JOIN order_value ov
ON o.order_id = ov.order_id
WHERE o.order_status = 'delivered'
GROUP BY purchase_month
ORDER BY purchase_month;
"""
run_sql(query)

,purchase_month,distinct_orders,total_gross_order_value,average_gross_order_value,average_item_count
0,2016-09-01,1,143.46,143.460000,3.000000
1,2016-10-01,265,46490.66,175.436453,1.181132
2,2016-12-01,1,19.62,19.620000,1.000000
3,2017-01-01,750,127482.37,169.976493,1.217333
4,2017-02-01,1653,271239.32,164.089123,1.124017
5,2017-03-01,2546,414330.95,162.738001,1.137863
6,2017-04-01,2303,390812.40,169.697091,1.115502
7,2017-05-01,3546,566851.40,159.856571,1.129160
8,2017-06-01,3135,490050.37,156.315907,1.112919
9,2017-07-01,3872,566299.08,146.254928,1.140496


In [ ]:
# one to many merge - merge orders and order items
orders_items_raw = orders.merge(
    order_items,
    on="order_id",
    how="left", # keep all orders even if there is no item for it - show it as NaN
    validate="one_to_many", # when there is a one_to_many relationship, python repeats order info for every order_item - makes the table order item level
    indicator="raw_item_match",
)


In [18]:
print("Rows in orders:", len(orders))
print("Rows after raw merge:", len(orders_items_raw))
print("Unique order IDs after raw merge:", orders_items_raw["order_id"].nunique())

orders_items_raw["raw_item_match"].value_counts()

Rows in orders: 99441
Rows after raw merge: 113425
Unique order IDs after raw merge: 99441


raw_item_match
both          112650
left_only        775
right_only         0
Name: count, dtype: int64

In [22]:
orders_items_raw.head(12)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,gross_item_value,raw_item_match
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,38.71,both
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,141.46,both
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,179.12,both
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,72.20,both
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,28.62,both
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,1.0,060cb19345d90064d1015407193c233d,8581055ce74af1daba164fdbd55a40de,2017-07-13 22:10:13,147.90,27.36,175.26,both
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,1.0,a1804276d9941ac0733cfd409f5206eb,dc8798cbf453b7e0f98745e396cc5616,2017-04-19 13:25:17,49.90,16.05,65.95,both
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,1.0,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,2017-05-22 13:22:11,59.99,15.17,75.16,both
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,1.0,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,2017-01-27 18:29:09,19.90,16.05,35.95,both
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,1.0,9a78fb9862b10749a117f7fc3c31f051,7c67e1448b00f6e969d365cea6b010ab,2017-08-11 12:05:32,149.99,19.77,169.76,both


In [24]:
# Merge orders with order_value.
# Requirements:
# join on order_id
# keep every order
# validate the relationship
# add a descriptive indicator column

orders_expanded = orders.merge(
    order_value,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator=True
)

orders_expanded.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,gross_order_value,item_count,avg_item_value,_merge
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,1.0,38.71,both
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,1.0,141.46,both
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,1.0,179.12,both
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,1.0,72.20,both
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,1.0,28.62,both


In [26]:
print("Rows in orders:", len(orders))
print("Rows in orders_expanded:", len(orders_expanded))
print("Unique order IDs:", orders_expanded["order_id"].nunique())

orders_expanded["_merge"].value_counts()

Rows in orders: 99441
Rows in orders_expanded: 99441
Unique order IDs: 99441


_merge
both          98666
left_only       775
right_only        0
Name: count, dtype: int64

In [ ]:
# Using orders_expanded, create a summary by order_status.
# Return:
# number of distinct orders
# total gross order value
# average gross order value
# Sort from highest to lowest total gross order value.
# For this question, leave missing gross_order_value values as missing.

status_summary = (
    orders_expanded
    .groupby('order_status',as_index=False)
    .agg(
        distinct_orders = ('order_id','nunique'),
        total_order_value = ('gross_order_value','sum'),
        average_order_value = ('gross_order_value','mean'),
    )
).sort_values('total_order_value',ascending=False)
status_summary.head(15)

# if we wanted to not have NaN values, we could have used:
# orders_expanded["gross_order_value_filled"] = orders_expanded["gross_order_value"].fillna(0)

,order_status,distinct_orders,total_order_value,average_order_value
3,delivered,96478,15419773.75,159.826839
6,shipped,1107,177129.34,160.153110
1,canceled,625,105885.72,229.687028
5,processing,301,69394.11,230.545216
4,invoiced,314,68988.75,221.117788
7,unavailable,609,2140.49,356.748333
0,approved,2,241.08,120.540000
2,created,5,0.00,NaN
